In [ ]:
import pandas as pd 
import io 
import json
pd.options.display.max_rows = 200

# helpers

In [ ]:
def _process_data_(file):
    with open(file, 'r') as file:
        log_content = file.read()
    sections = log_content.split('Sandbox logs:')[1].split('Activities log:')
    sandbox_log =  sections[0].strip()
    activities_log = sections[1].split('Trade History:')[0]
    # sandbox_log_list = [json.loads(line) for line in sandbox_log.split('\n')]
    trade_history =  json.loads(sections[1].split('Trade History:')[1])
    # sandbox_log_df = pd.DataFrame(sandbox_log_list)
    market_data_df = pd.read_csv(io.StringIO(activities_log), sep=";", header=0)
    trade_history_df = pd.json_normalize(trade_history)
    return market_data_df, trade_history_df

In [ ]:
def get_prev_returns(df, col, its):
    prev_col = f"{col}_prev_{its}_its"
    df[prev_col] = df[col].shift(its)
    df[f"{col}_returns_from_{its}_its_ago"] = (df[col] - df[prev_col]) / df[prev_col]
    df.drop(columns=[prev_col], inplace=True)
    return df

def get_future_returns(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    df[f"{col}_returns_in_{its}_its"] = (df[future_col] - df[col]) / df[col]
    df.drop(columns=[future_col], inplace=True)
    return df

def get_centered_returns(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    prev_col = f"{col}_prev_{its}_its"
    df[prev_col] = df[col].shift(its)
    df[f"{col}_returns_centered_with_{its}_its"] = (df[future_col] - df[prev_col])/df[prev_col]
    df.drop(columns=[prev_col], inplace=True)
    df.drop(columns=[future_col], inplace=True)
    return df

# process 23 data

In [ ]:
day = 3

In [ ]:
df = pd.read_csv(f"2023_data_logs/r{day}.csv", sep=';')

In [ ]:
df_diving = df[df['product'] == 'DIVING_GEAR']

In [ ]:
df_diving.columns

In [ ]:
df_pred = df_diving[['day','timestamp', 'mid_price']].copy()

In [ ]:
df_pred = get_future_returns(df_pred, 'mid_price', 1).reset_index(drop=True)
df_pred = get_future_returns(df_pred, 'mid_price', 2).reset_index(drop=True)

In [ ]:
df_pred['ROSES_pred_returns_in_1_its'] = df_pred['mid_price_returns_in_1_its']* 3.07

In [ ]:
df_pred['ROSES_pred_returns_in_1_its'].abs().describe()

In [ ]:
df_pred.columns

In [ ]:
df_pred[['ROSES_pred_returns_in_1_its']].to_csv('roses_pred_returns_r5.csv', index=False)

In [ ]:
df_pred

In [ ]:
pred_returns_list = df_pred['ROSES_pred_returns_in_1_its'].tolist()

In [ ]:
import numpy as np
roses_pred_price = [14411.5]
for i in pred_returns_list:
    roses_pred_price.append(roses_pred_price[-1] * (1 + i))
rose_pred_returns_price = np.diff(roses_pred_price)
df_pred['ROSES_pred_returns_in_1_its_price'] = np.round( 2 * rose_pred_returns_price)/2

In [ ]:
pred_returns_pct_list = df_pred['ROSES_pred_returns_in_1_its'].tolist()
pred_returns_price_list = df_pred['ROSES_pred_returns_in_1_its_price'].tolist()

In [ ]:
pred_returns_price_agg = []
sign = 0
sum_return = 0
for i in range(len(pred_returns_price_list)-1, -1 ,-1):
    if np.sign(pred_returns_price_list[i]) != sign:
        sum_return = 0
    sum_return += pred_returns_price_list[i]
    pred_returns_price_agg.append(sum_return)
    sign = np.sign(pred_returns_price_list[i])
pred_returns_price_agg = pred_returns_price_agg[::-1]
df_pred['pred_returns_price_agg'] = pred_returns_price_agg

In [ ]:
pred_returns_pct_agg = []
sign = 0
prod_return = 1
for i in range(len(pred_returns_pct_list)-1, -1 ,-1):
    if np.sign(pred_returns_pct_list[i]) != sign:
        prod_return = 1
    prod_return *= 1 + pred_returns_pct_list[i]
    pred_returns_pct_agg.append(prod_return - 1)
    sign = np.sign(pred_returns_pct_list[i])
pred_returns_pct_agg = pred_returns_pct_agg[::-1]
df_pred['pred_returns_pct_agg'] = pred_returns_pct_agg

In [ ]:
df_pred.head(30)

In [ ]:
trades_list = []
for ret in pred_returns_price_agg:
    if ret > 0.5 and ret <= 1.5:
        trades_list.append('b')
    elif ret > 1.5:
        trades_list.append("B")
    elif ret < -0.5 and ret >= -1.5: 
        trades_list.append('s')
    elif ret < -1.5:
        trades_list.append("S")
    else:
        trades_list.append('h')

In [ ]:
trades_string = ''.join(trades_list)

In [ ]:
trades_string

In [ ]:
day = 2
df = pd.read_csv(f"2023_data_logs/r{day}.csv", sep=';')
df_coconut = df[df['product'] == 'COCONUTS']
df_pred_coconut = df_coconut[['day','timestamp', 'mid_price']].copy()
df_pred_coconut = get_future_returns(df_pred_coconut, 'mid_price', 1).reset_index(drop=True)
df_pred_coconut.tail()

In [ ]:
df_coconut

In [ ]:
df_r4, _ = _process_data_("./2024_data_logs/results_round4.log")
df_r4_coconuts = df_r4[df_r4['product'] == 'COCONUT'].copy().reset_index(drop=True)
# Calculate the count of bid and ask prices with the price difference to the mid price
# df_r4_coconuts['mid_price'] = (df_r4_coconuts['bid_price_1'] + df_r4_coconuts['ask_price_1']) / 2

# Initialize dictionaries to store the sum of volumes for each unique price difference
bid_price_diff_volumes = {}
ask_price_diff_volumes = {}

# Iterate through each row to calculate differences and sum volumes for unique differences
for index, row in df_r4_coconuts.iterrows():
    bid_prices = [row['bid_price_1'], row['bid_price_2'], row['bid_price_3']]
    ask_prices = [row['ask_price_1'], row['ask_price_2'], row['ask_price_3']]
    bid_volumes = [row['bid_volume_1'], row['bid_volume_2'], row['bid_volume_3']]
    ask_volumes = [row['ask_volume_1'], row['ask_volume_2'], row['ask_volume_3']]
    
    # Process bid prices and volumes
    for i in range(3):
        if pd.notna(bid_prices[i]) and pd.notna(bid_volumes[i]):
            bid_diff = abs(bid_prices[i] - row['mid_price'])
            if bid_diff in bid_price_diff_volumes:
                bid_price_diff_volumes[bid_diff] += bid_volumes[i]
            else:
                bid_price_diff_volumes[bid_diff] = bid_volumes[i]
    
    # Process ask prices and volumes
    for i in range(3):
        if pd.notna(ask_prices[i]) and pd.notna(ask_volumes[i]):
            ask_diff = abs(ask_prices[i] - row['mid_price'])
            if ask_diff in ask_price_diff_volumes:
                ask_price_diff_volumes[ask_diff] += ask_volumes[i]
            else:
                ask_price_diff_volumes[ask_diff] = ask_volumes[i]

# Display the dictionaries containing the sum of volumes for each unique price difference
print("Bid Price Difference Volumes:", bid_price_diff_volumes)
print("Ask Price Difference Volumes:", ask_price_diff_volumes)


In [ ]:
# Calculate the total weighted spread for bid and ask price difference volumes
if bid_price_diff_volumes:
    total_bid_volume = sum(bid_price_diff_volumes.values())
    weighted_bid_spread = sum(diff * volume for diff, volume in bid_price_diff_volumes.items()) / total_bid_volume if total_bid_volume else 0
else:
    weighted_bid_spread = 0

if ask_price_diff_volumes:
    total_ask_volume = sum(ask_price_diff_volumes.values())
    weighted_ask_spread = sum(diff * volume for diff, volume in ask_price_diff_volumes.items()) / total_ask_volume if total_ask_volume else 0
else:
    weighted_ask_spread = 0

print("Total Weighted Bid Price Spread:", weighted_bid_spread)
print("Total Weighted Ask Price Spread:", weighted_ask_spread)


In [ ]:
coconut_past_price = df_pred_coconut['mid_price'].to_numpy()
spread = [(0.9894962494138442 + 0.9880513910522672)/2] * len(coconut_past_price)


In [ ]:
def optimal_trading_dp(prices, spreads):
    n = len(prices)
    dp = [[float('-inf')] * 7 for _ in range(n)]  # From -3 to 3, 7 positions
    action = [[''] * 7 for _ in range(n)]  # To store actions

    # Initialize the starting position (no stock held)
    dp[0][3] = 0  # Start with no position, Cash is 0
    action[0][3] = ''  # No action at start

    for i in range(1, n):
        for j in range(0, 7):
            # Calculate PnL for holding, buying, or selling
            hold = dp[i-1][j] if dp[i-1][j] != float('-inf') else float('-inf')
            buy = dp[i-1][j-1] - prices[i-1] - spreads[i-1] if j > 0 else float('-inf')
            sell = dp[i-1][j+1] + prices[i-1] - spreads[i-1] if j < 6 else float('-inf')

            # Choose the action with the highest PnL
            hold_pnl = hold + (j - 3) * prices[i]
            buy_pnl = buy + (j - 3) * prices[i]
            sell_pnl = sell + (j - 3) * prices[i]
            
            # print(hold_pnl, buy_pnl, sell_pnl)
            best_action = max(hold_pnl, buy_pnl, sell_pnl)
            if best_action == hold_pnl:
                dp[i][j] = hold
            elif best_action == buy_pnl:
                dp[i][j] = buy
            else:
                dp[i][j] = sell

            if best_action == hold_pnl:
                action[i][j] = 'h'
            elif best_action == buy_pnl:
                action[i][j] = 'b'
            else:
                action[i][j] = 's'
    # Backtrack to find the sequence of actions
    trades_list = []
    # Start from the position with maximum PnL at time n-1
    pnl = np.array(dp[n-1]) + (np.arange(-3,4) * prices[n-1])
    current_position = np.argmax(pnl)
    for i in range(n-1, -1, -1):
        trades_list.append(action[i][current_position])
        if action[i][current_position] == 'b':
            current_position -= 1
        elif action[i][current_position] == 's':
            current_position += 1

    trades_list.reverse()
    trades_list.append('h')
    return trades_list, pnl[np.argmax(pnl)]  # Return the actions and the maximum PnL

# Example usage
trades, max_pnl = optimal_trading_dp(coconut_past_price, spread)
# print(trades)
print("Max PnL:", max_pnl)

In [ ]:
trades_string = ''.join(trades)
# len(trades_string)
trades_string

In [ ]:
df_pred_coconut24 = [9883.5]
for i in df_pred_coconut['mid_price_returns_in_1_its']:
    if pd.isna(i):
        df_pred_coconut24.append(df_pred_coconut24[-1])
    else:
        df_pred_coconut24.append(df_pred_coconut24[-1] * (1 + i))
df_pred_coconut['COCONUT_pred_price'] = np.array(df_pred_coconut24[:-1])
df_pred_coconut['COCONUT_pred_price_diff'] = df_pred_coconut['COCONUT_pred_price'].diff().shift(-1)
df_pred_coconut['COCONUT_actual_price'] = df_r4_coconuts['mid_price']


In [ ]:
pred_returns_price_list = df_pred_coconut['COCONUTS_price_diff'].to_numpy()
pred_returns_price_agg = []
sign = 0
sum_return = 0
for i in range(len(pred_returns_price_list)-1, -1 ,-1):
    if np.sign(pred_returns_price_list[i]) != sign:
        sum_return = 0
    sum_return += pred_returns_price_list[i]
    pred_returns_price_agg.append(sum_return)
    sign = np.sign(pred_returns_price_list[i])
pred_returns_price_agg = pred_returns_price_agg[::-1]
df_pred_coconut['COCONUT_pred_price_agg'] = pred_returns_price_agg

In [ ]:
trades_list = []
for ret in pred_returns_price_agg:
    if ret > 0.5 and ret <= 2:
        trades_list.append('b')
    elif ret > 2:
        trades_list.append("B")
    elif ret < -0.5 and ret >= -2: 
        trades_list.append('s')
    elif ret < -2:
        trades_list.append("S")
    else:
        trades_list.append('h')

In [ ]:
trades_string = ''.join(trades_list)
trades_string